# HAGI - Seed-Stability Gate (B vs D)

The 4-model ablation gave **D < B by ~0.015 nats** - a small gap from a single run
each, so it could be seed luck (random init + data order) rather than the
architecture. This notebook re-trains **B and D across several seeds** and checks
whether **D < B holds every time** on a **held-out shard**.

- All four originals are untouched; this is a separate, self-contained experiment.
- Resumable: re-run cell 5 after any session death - completed runs are skipped.
- No HF token needed (checkpoints stay on Drive).

**Before running:** Runtime -> Change runtime type -> **A100 GPU**. The original
Stage-0 shards must be on Drive at `DATA` below.

> **Budget:** each seed trains BOTH B and D. ~0.44 A100-min per million tokens for
> the pair. 5 seeds x 120M ~= **~57 units / ~4.5h**. Tune `SEEDS` / `SEED_TOKENS`
> in cell 4 and check the printed estimate **before** running cell 5.

## 1. Clone + update the repo

In [ ]:
import os
%cd /content
if not os.path.isdir('HAGI'):
    !git clone -b experimental https://github.com/ShmidtS/HAGI.git
%cd /content/HAGI
!git pull --ff-only origin experimental   # needs the --seed and --json flags
print('cwd:', os.getcwd())

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Drive + knobs + preflight
Mounts Drive, builds the held-out split (6 shards train / 1 shard val), and prints
the unit estimate. **Read the estimate before running cell 5.**

In [ ]:
import glob, torch
from google.colab import drive
drive.mount('/content/drive')

# ---- knobs (WATCH BUDGET) ------------------------------------------------
DATA        = '/content/drive/MyDrive/hagi-data'    # original 7 Stage-0 shards
CKPT_ROOT   = '/content/drive/MyDrive/hagi-seeds'   # separate from the originals
SEEDS       = [1, 2, 3, 4, 5]
SEED_TOKENS = 120_000_000                           # per run; lower = cheaper
# --------------------------------------------------------------------------

RESULTS_JSON = f'{CKPT_ROOT}/seed_results.json'
TPS     = 16 * 4 * 1024
STEPS   = SEED_TOKENS // TPS + 50
MAXSTEP = -(-SEED_TOKENS // TPS)
os.makedirs(CKPT_ROOT, exist_ok=True)

assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> A100.'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name} | {p.total_memory/1e9:.0f} GB')

shards = sorted(glob.glob(f'{DATA}/*.bin'))
assert len(shards) >= 2, f'need >=2 shards in {DATA}, found {len(shards)}'
TRAIN_DIR, VAL_DIR = '/content/hagi_train', '/content/hagi_val'
for d in (TRAIN_DIR, VAL_DIR):
    os.makedirs(d, exist_ok=True)
    for f in glob.glob(f'{d}/*.bin'): os.remove(f)
for s in shards[:-1]: os.symlink(s, f'{TRAIN_DIR}/{os.path.basename(s)}')
os.symlink(shards[-1], f'{VAL_DIR}/{os.path.basename(shards[-1])}')
VAL_NAME = os.path.basename(shards[-1])

est_min  = 0.44 * SEED_TOKENS / 1e6 * len(SEEDS)
print(f'\ntrain: {len(shards)-1} shards | held-out val: {VAL_NAME}')
print(f'plan: seeds {SEEDS} x (B,D) @ {SEED_TOKENS:,} tok (~{MAXSTEP} steps each)')
print(f'EST COST: ~{est_min:.0f} A100-min  (~{est_min*0.217:.0f} units). Tune knobs if too high.')

## 4. Run all seeds + verdict
Trains B then D for each seed on the 6-shard train set, evaluates both on the
held-out shard, and prints `D < B?` per seed plus the overall gate. Resumable -
re-run this cell to continue; finished runs are skipped.

In [ ]:
import json, subprocess, re
REPO = '/content/HAGI'

def done(d):
    cks = glob.glob(f'{d}/step-*.pt')
    return bool(cks) and max(int(re.search(r'step-(\d+)', c).group(1)) for c in cks) >= MAXSTEP

def train(model, seed):
    d = f'{CKPT_ROOT}/s{seed}/ablation_{model}'
    if done(d):
        print(f'  [{model} s{seed}] complete -> skip'); return
    subprocess.run(['python','-u','-m','prototype.training.train',
        '--config', f'configs/ablation_{model}.yaml', '--data', TRAIN_DIR, '--device','cuda',
        '--ckpt-dir', f'{CKPT_ROOT}/s{seed}', '--seed', str(seed),
        '--train-tokens', str(SEED_TOKENS), '--steps', str(STEPS), '--resume','auto'],
        cwd=REPO, check=True)

def latest(model, seed):
    cks = sorted(glob.glob(f'{CKPT_ROOT}/s{seed}/ablation_{model}/step-*.pt'))
    return cks[-1] if cks else None

def eval_pair(seed):
    r = subprocess.run(['python','-u','scripts/eval_loss.py','--data',VAL_DIR,'--device','cuda',
        '--batches','50','--json','--ckpt', latest('b',seed), latest('d',seed)],
        cwd=REPO, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0: print('STDERR', r.stderr); return None, None
    line = next((l for l in r.stdout.splitlines() if l.startswith('JSON ')), None)
    j = json.loads(line[5:]) if line else {}
    return j.get('ablation_b'), j.get('ablation_d')

results = json.load(open(RESULTS_JSON)) if os.path.exists(RESULTS_JSON) else {}
for seed in SEEDS:
    print(f'===== SEED {seed} =====', flush=True)
    train('b', seed); train('d', seed)
    b, d = eval_pair(seed)
    if b and d:
        results[str(seed)] = {'b': b['loss'], 'd': d['loss'], 'd_minus_b': round(d['loss']-b['loss'],6)}
        json.dump(results, open(RESULTS_JSON,'w'), indent=2)

print('\n================ SEED-STABILITY SUMMARY ================')
print(f'held-out val: {VAL_NAME} | {SEED_TOKENS:,} tokens/run')
print(f"{'seed':>4} {'B loss':>9} {'D loss':>9} {'D-B':>9}  D<B?")
wins, deltas = 0, []
for s in SEEDS:
    r = results.get(str(s))
    if not r: print(f'{s:>4}   (missing)'); continue
    w = r['d_minus_b'] < 0; wins += w; deltas.append(r['d_minus_b'])
    print(f"{s:>4} {r['b']:>9.4f} {r['d']:>9.4f} {r['d_minus_b']:>+9.4f}  {'YES' if w else 'no'}")
if deltas:
    mean = sum(deltas)/len(deltas)
    print(f'\nD beat B in {wins}/{len(deltas)} seeds | mean D-B = {mean:+.4f}')
    print('GATE PASSED - seed-robust.' if wins==len(deltas) else 'MIXED - do NOT scale on this.')
    print(f"\nPaste to Claude: SEED_STABILITY = dict(seeds={SEEDS}, d_below_b={wins}, "
          f"mean_d_minus_b={round(mean,4)}, val='{VAL_NAME}')")